# Abstraction in Python

Goal: understand why abstraction exists, abstract base classes, `@abstractmethod`, and how abstraction differs from encapsulation.

## 1. What is Abstraction?

**Why we need it:** to drive a car, you press the accelerator — you don't need to know how fuel injection, ignition timing, or the transmission actually work underneath. The dashboard shows you exactly what you need (speed, fuel level) and hides everything else. If you had to think about every internal mechanical detail every time you drove, it would be unusable. Software has the same problem: if every piece of code that uses a class has to understand and depend on *how* it works internally, any change to those internals breaks everything that uses it.

**Use case:** designing any class where callers should depend only on *what* it does, not *how* — a `PaymentProcessor` that callers just call `.process_payment(amount)` on, without knowing or caring whether it's talking to Stripe, PayPal, or a bank API underneath; a `Database` class where callers call `.save(record)` without knowing whether it's backed by Postgres or a JSON file.

**How to implement:** expose a small, clear set of methods that describe *what* the class can do — its interface — and keep everything about *how* it does that inside the class, not depended upon from outside. Python has a specific tool for defining and enforcing this kind of interface across multiple related classes: **abstract base classes**, covered next.

## 2. The Problem: Nothing Enforces the Interface

**Why we need it:** you could try to define an interface just by convention — "every payment processor must have a `process_payment` method" — and rely on developers remembering that rule. But Python won't stop someone from writing a subclass that forgets to implement it. The mistake isn't caught until the moment that specific method is actually called, possibly in production, long after the class was written.

**Use case:** any time you're designing a base class meant purely as a shared contract for several subclasses (not meant to be used on its own) — you want a mistake like "forgot to implement `process_payment`" caught immediately, ideally when the broken class is even instantiated, not buried in a runtime code path that might not run until later.

**How it shows up:** the example below defines a plain (non-abstract) base class as a "contract," then a subclass that forgets to implement it. Nothing stops you from creating an instance of the broken subclass — the mistake only surfaces once you actually try to call the missing method.

In [1]:
class PaymentProcessor:
    """Meant as a contract: 'subclasses should implement process_payment'.
    But this is just a comment -- Python doesn't enforce it."""
    def process_payment(self, amount):
        raise NotImplementedError   # a manual, easy-to-forget convention


class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount):
        return f"Charged ${amount} to credit card"


class BrokenProcessor(PaymentProcessor):
    """Oops -- forgot to override process_payment."""
    def refund(self, amount):
        return f"Refunded ${amount}"


cc = CreditCardProcessor()
print(cc.process_payment(100))

broken = BrokenProcessor()          # this line runs perfectly fine -- no warning at all
print("BrokenProcessor was created without any error")

try:
    broken.process_payment(50)      # the mistake only surfaces here, at call time
except NotImplementedError:
    print("NotImplementedError: only discovered now, way too late")

Charged $100 to credit card
BrokenProcessor was created without any error
NotImplementedError: only discovered now, way too late


## 3. Abstract Base Classes: Enforcing the Contract

**Why we need it:** the previous section's failure mode — a broken subclass instantiates fine and only fails later — is exactly what Python's `abc` module exists to prevent. It moves the error from "whenever this method happens to get called" to "the moment you try to create the broken object at all."

**Use case:** the exact same `PaymentProcessor` scenario, but now with the missing-method mistake caught immediately, before `BrokenProcessor()` is ever allowed to run.

**How to implement:**
1. Import `ABC` and `abstractmethod` from the `abc` module.
2. Make the base class inherit from `ABC` instead of nothing (or `object`).
3. Mark any method that subclasses *must* implement with `@abstractmethod`, and give it just a `pass` or a docstring — it's never meant to actually run.

Two consequences follow automatically: you can no longer create an instance of the abstract base class itself (`PaymentProcessor()` becomes an error, since it's a contract, not a usable class on its own), and any subclass that doesn't implement every abstract method also can't be instantiated — Python raises the error the instant you try, not later when the method is finally called.

In [2]:
from abc import ABC, abstractmethod


class PaymentProcessor(ABC):           # inherits from ABC, not just object
    @abstractmethod
    def process_payment(self, amount):
        """Every subclass MUST implement this."""
        pass


class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount):
        return f"Charged ${amount} to credit card"


class BrokenProcessor(PaymentProcessor):
    """Still forgets process_payment -- but now that's caught immediately."""
    def refund(self, amount):
        return f"Refunded ${amount}"


cc = CreditCardProcessor()
print(cc.process_payment(100))

try:
    PaymentProcessor()          # can't instantiate the abstract base class itself
except TypeError as e:
    print(f"TypeError: {e}")

try:
    broken = BrokenProcessor()  # fails HERE now, at creation -- not at some later call
except TypeError as e:
    print(f"TypeError: {e}")

Charged $100 to credit card
TypeError: Can't instantiate abstract class PaymentProcessor without an implementation for abstract method 'process_payment'
TypeError: Can't instantiate abstract class BrokenProcessor without an implementation for abstract method 'process_payment'


## 4. Mixing Abstract and Concrete Methods

**Why we need it:** an abstract base class doesn't have to be *only* a list of unimplemented requirements. Usually some behavior really is identical across every subclass (logging, validation, a shared helper), while only a specific part genuinely differs per subclass. Forcing every subclass to also reimplement the shared part would bring back the duplication problem inheritance was meant to solve in the first place.

**Use case:** every `Shape` subclass needs its own `area()` formula (must vary), but `describe()` — which just formats a message using whatever `area()` returns — is identical for all of them (should not vary, and shouldn't be copy-pasted into every subclass).

**How to implement:** in the same `ABC` subclass, mark only the methods that truly must differ per subclass with `@abstractmethod`, and write any shared logic as an ordinary method with a real body. Concrete methods on an abstract base class can freely call the abstract ones — by the time they actually run, on some concrete subclass instance, the abstract method is guaranteed to have a real implementation.

In [3]:
class Shape(ABC):
    @abstractmethod
    def area(self):                  # must differ per subclass -- no sensible default
        pass

    def describe(self):              # shared for every subclass -- written once, here
        return f"{type(self).__name__} with area {self.area():.2f}"


class Circle(Shape):
    def __init__(self, radius):
        self.radius = radius

    def area(self):
        return 3.14159 * self.radius ** 2


class Square(Shape):
    def __init__(self, side):
        self.side = side

    def area(self):
        return self.side ** 2


for shape in [Circle(3), Square(4)]:
    print(shape.describe())   # describe() is never redefined, but still uses each shape's own area()

Circle with area 28.27
Square with area 16.00


## 5. Abstraction vs Encapsulation — They're Not the Same Thing

These two get mixed up constantly because they both involve "hiding" something, but they hide different things for different reasons.

- **Encapsulation** (previous notebook) is about hiding *data and implementation details* to protect an object's internal state from being corrupted — `self._balance` plus a validating `@property` so `balance` can never go negative. It's a design-level concern about one specific class's internals.
- **Abstraction** is about hiding *complexity* by exposing only the essential *behavior* — deciding that callers only need to know a processor can `process_payment(amount)`, not how it talks to a card network. It's a higher-level concern about the shape of an interface shared across multiple classes.

A useful shorthand: encapsulation protects *how data is stored*; abstraction simplifies *what operations are exposed*. In practice they usually show up together — an abstract base class defines *what* subclasses must do, and each subclass then uses encapsulation internally to protect *how* it does it.

## 6. Putting It Together: Payment Processors, Abstraction + Encapsulation

**Why we need it:** to see the shorthand from section 5 actually happen in code — an abstract contract at the top, each subclass free to protect its own internal details however it needs to, and calling code that only ever touches the abstract interface.

**Use case:** a checkout flow that needs to accept different payment methods, each with different internal validation rules, without the checkout code needing an `if/elif` per payment type.

**How it's implemented here:** `PaymentProcessor` is the abstract contract — `process_payment(amount)` must be implemented by every subclass. `CreditCardProcessor` encapsulates its own internal `_validate` check as a protected method; `PayPalProcessor` encapsulates a completely different internal check. The `checkout` function at the bottom only ever calls `processor.process_payment(amount)` — it has no idea, and doesn't need to know, what either subclass is doing internally.

In [4]:
class PaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount):
        pass


class CreditCardProcessor(PaymentProcessor):
    def __init__(self, card_number):
        self._card_number = card_number         # encapsulated: internal detail

    def _validate(self, amount):                # encapsulated: internal helper, protected
        return amount <= 5000 and len(self._card_number) == 16

    def process_payment(self, amount):
        if not self._validate(amount):
            return "Credit card payment declined"
        return f"Charged ${amount} to card ending in {self._card_number[-4:]}"


class PayPalProcessor(PaymentProcessor):
    def __init__(self, email):
        self._email = email                      # different internals entirely

    def _validate(self, amount):
        return "@" in self._email and amount <= 10000

    def process_payment(self, amount):
        if not self._validate(amount):
            return "PayPal payment declined"
        return f"Charged ${amount} via PayPal account {self._email}"


def checkout(processor, amount):
    # this function only knows about the abstract interface -- process_payment(amount)
    return processor.process_payment(amount)


processors = [
    CreditCardProcessor("1234567812345678"),
    PayPalProcessor("alice@example.com"),
]

for processor in processors:
    print(checkout(processor, 200))

Charged $200 to card ending in 5678
Charged $200 via PayPal account alice@example.com
